In [ ]:
pip install ucimlrepo

In [35]:
import pandas as pd

from ucimlrepo import fetch_ucirepo

# fetch dataset
breast_cancer = fetch_ucirepo(id=14)

# data (as pandas dataframes)
X = breast_cancer.data.features
y = breast_cancer.data.targets

# metadata
print(breast_cancer.metadata)

# variable information
print(breast_cancer.variables)


{'uci_id': 14, 'name': 'Breast Cancer', 'repository_url': 'https://archive.ics.uci.edu/dataset/14/breast+cancer', 'data_url': 'https://archive.ics.uci.edu/static/public/14/data.csv', 'abstract': 'This breast cancer domain was obtained from the University Medical Centre, Institute of Oncology, Ljubljana, Yugoslavia. This is one of three domains provided by the Oncology Institute that has repeatedly appeared in the machine learning literature. (See also lymphography and primary-tumor.)', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 286, 'num_features': 9, 'feature_types': ['Categorical'], 'demographics': ['Age'], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1988, 'last_updated': 'Thu Mar 07 2024', 'dataset_doi': '10.24432/C51P4M', 'creators': ['Matjaz Zwitter', 'Milan Soklic'], 'intro_paper': None, 'additional_info': {'summary': 'Thi

In [36]:
X['class'] = y
print(X.head())

     age menopause tumor-size inv-nodes node-caps  deg-malig breast  \
0  30-39   premeno      30-34       0-2        no          3   left   
1  40-49   premeno      20-24       0-2        no          2  right   
2  40-49   premeno      20-24       0-2        no          2   left   
3  60-69      ge40      15-19       0-2        no          2  right   
4  40-49   premeno        0-4       0-2        no          2  right   

  breast-quad irradiat                 class  
0    left_low       no  no-recurrence-events  
1    right_up       no  no-recurrence-events  
2    left_low       no  no-recurrence-events  
3     left_up       no  no-recurrence-events  
4   right_low       no  no-recurrence-events  


In [37]:
#Example of melt function, where irradiat and class are turned into variable values,
#with age serving as the identifier

X2 = X.copy()
X2 = X2.melt(id_vars = ['age', 'menopause'], value_vars = ['irradiat', 'class'])
print(X2.head())

     age menopause  variable value
0  30-39   premeno  irradiat    no
1  40-49   premeno  irradiat    no
2  40-49   premeno  irradiat    no
3  60-69      ge40  irradiat    no
4  40-49   premeno  irradiat    no


In [58]:
#Example of groupby and aggregate used together. Used to get a quick glance at the
#relationship between tumor size and recurrence.

X3 = X.copy()
X3 = X3.groupby(['tumor-size'])['class'].aggregate(['value_counts'])
print(X3)

                                 value_counts
tumor-size class                             
0-4        no-recurrence-events             7
           recurrence-events                1
14-Oct     no-recurrence-events            27
           recurrence-events                1
15-19      no-recurrence-events            23
           recurrence-events                7
20-24      no-recurrence-events            34
           recurrence-events               16
25-29      no-recurrence-events            36
           recurrence-events               18
30-34      no-recurrence-events            35
           recurrence-events               25
35-39      no-recurrence-events            12
           recurrence-events                7
40-44      no-recurrence-events            16
           recurrence-events                6
45-49      no-recurrence-events             2
           recurrence-events                1
50-54      no-recurrence-events             5
           recurrence-events      

In [71]:
#Uses iterate, groupby and aggregate functions to turn tumor size into a numerical
#variable, and then summarize the relationship between deg-malig and mean tumor size (using the lower bound for size).

X4 = X.copy()

#I don't know if tumor-size = 9-may / 14-oct is a mistake for means something,
#but I'm deleting those values

X4.drop(X4[X4['tumor-size'] == '9-May'].index, inplace=True)
X4.drop(X4[X4['tumor-size'] == '14-Oct'].index, inplace=True)

for index, row in X4.iterrows():
  X4.loc[index, 'tumor-size'] = int(row['tumor-size'][:row['tumor-size'].index('-')])

print(X4['tumor-size'].unique())

print(X4.groupby('deg-malig')['tumor-size'].aggregate('mean'))

[30 20 15 0 25 50 40 35 45]
deg-malig
1    24.818182
2    25.818966
3    27.951807
Name: tumor-size, dtype: object


In [77]:
#Uses pivot_table function to aggregate menopause categories by value counts, then creates
#a new data frame which expresses those value counts as a function of tumor-size,
#and recurrence.

X5 = X.copy().pivot_table(index=['class'],
                          columns=['menopause'],
                          values = ['tumor-size'],
                          aggfunc = 'value_counts')
X5.fillna(0, inplace=True)
print(X5)

menopause                        ge40  lt40  premeno
class                tumor-size                     
no-recurrence-events 0-4          4.0   0.0      3.0
                     14-Oct      12.0   1.0     14.0
                     15-19       14.0   2.0      7.0
                     20-24       14.0   0.0     20.0
                     25-29       14.0   0.0     22.0
                     30-34       17.0   2.0     16.0
                     35-39        4.0   0.0      8.0
                     40-44        9.0   0.0      7.0
                     45-49        1.0   0.0      1.0
                     50-54        3.0   0.0      2.0
                     9-May        2.0   0.0      2.0
recurrence-events    0-4          0.0   0.0      1.0
                     14-Oct       1.0   0.0      0.0
                     15-19        1.0   0.0      6.0
                     20-24        9.0   2.0      5.0
                     25-29        5.0   0.0     13.0
                     30-34       11.0   0.0   